# 02. Limpieza y lematización del corpus

Filtra los posteos extraídos para armar el corpus de modelado: saca las notas sin cuerpo de texto, descarta versiones republicadas viejas (`-N/`), filtra idiomas extranjeros (dejando `es` y los falsos positivos en `pt`) y arma `document_text_extended` sumando el título, el copete, la bajada (`tagline`) y el cuerpo. Después lematiza todo por lotes con spaCy (`es_core_news_md`) corrigiendo formas como "países" -> "país".

**Entradas:** 
* `data/processed/posteos_anfibia.csv`

**Salidas:** 
* `data/processed/corpus_anfibia_limpio.csv` 
* `data/processed/corpus_anfibia_lematizado.csv`

---

In [1]:
from pathlib import Path
import pandas as pd

pd.set_option("display.max_colwidth", 120)

# Resolución estándar del proyecto
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_PATH = PROJECT_ROOT / "data" / "processed"
DATA_PATH = PROCESSED_PATH / "posteos_anfibia.csv"

assert DATA_PATH.exists(), f"Ejecutar primero: python src/extract_posteos_anfibia.py (Falta {DATA_PATH.name})"

df = pd.read_csv(DATA_PATH)
df["post_date"] = pd.to_datetime(df["post_date"], errors="coerce") 

print(f"Posts cargados: {len(df):,}")
print(f"Columnas: {list(df.columns)}")

Posts cargados: 3,622
Columnas: ['ID', 'source_url', 'guid', 'post_name', 'post_title', 'title_text', 'post_status', 'post_type', 'post_author', 'post_date', 'post_date_gmt', 'post_modified', 'post_modified_gmt', 'post_date_calendar', 'post_year', 'post_month', 'post_year_month', 'post_quarter', 'post_year_quarter', 'post_weekday', 'days_to_modified', 'comment_count', 'post_excerpt', 'excerpt_text', 'post_content', 'body_text', 'document_text', 'word_count', 'copete', 'tagline', 'autor_ids', 'autor_id_list', 'autor_names', 'illustrator_ids', 'illustrator_id_list', 'illustrator_names', 'thumbnail_id', 'yoast_metadesc', 'yoast_focuskw', 'yoast_reading_time_minutes', 'category', 'destacado', 'destacado-arriba', 'destacado-medio', 'etiquetas-de-recomendados', 'home-slider', 'nav_menu', 'post_format', 'post_tag', 'recomendado', 'slider-en-home', 'spnl_log_type', 'super-destacado', 'text_type', 'tipo-de-cronica', 'web_story_category', 'web_story_media_source', 'web_story_tag', 'wp_pattern_ca

### 1. Posts con body_text vacío

El script de exploracion (posteos_anfibia_exploracion.ipynb) reportó 26 posts sin cuerpo de texto . Hay que ver qué son antes de decidir si se descartan.

repaso de las variables relevantes, considerandose "limpio" cuando el texto no contiene HTML, bloques de Gutenberg, shortcodes ni entidades HTML
- title_text: es el titulo limpio 
- body_text: es el cuerpo del texto limpio
- excerpt_text: es el extracto limpio 
- document_text: es el conjunto de title_text + body_text
- word count: cantidad de palabras de document_text
- copete: es un anticipo sobre que trata el post
- tagline: es una frase de presentacion 

In [2]:
empty_body = df[df["body_text"].fillna("").str.strip().eq("")]
print(f"Posts con body_text vacío: {len(empty_body)}")
display(empty_body[["ID", "post_date", "title_text","word_count", "source_url"]].sort_values("post_date"))

Posts con body_text vacío: 13


,ID,post_date,title_text,word_count,source_url
148,1755,2014-06-21 19:37:08,"Por lo menos así, lo veo yo",7,https://www.revistaanfibia.com/la-critica-anfibia/
274,14888,2014-10-31 10:59:18,El nadador,2,https://www.revistaanfibia.com/el-nadador/
431,24813,2015-06-06 05:56:40,"Venganza, judíos",2,https://www.revistaanfibia.com/venganza-judios/
564,30664,2015-10-09 07:43:01,Test peronista,2,https://www.revistaanfibia.com/test-peronista/
1404,94370,2018-01-29 11:18:53,El héroe del Mediterráneo,4,https://www.revistaanfibia.com/heroe-mediterraneo/
1549,103460,2018-06-06 16:59:47,La ciudad de las mujeres,5,https://www.revistaanfibia.com/la-ciudad-de-las-mujeres/
1554,103807,2018-06-11 10:45:48,La vida en verde,4,https://www.revistaanfibia.com/la-vida-en-verde/
1560,123696,2018-06-13 11:17:19,Desayuno con autoras,3,https://www.revistaanfibia.com/desayuno-con-autoras/
1566,123539,2018-06-15 11:51:14,Las figus de las pioneras,5,https://www.revistaanfibia.com/las-figus-de-las-pioneras/
1661,94860,2018-09-05 15:46:08,Gente que sí: Miguel Rep,5,https://www.revistaanfibia.com/gente-miguel-rep/


Ingresando manualmente a los liks se encuentra que estos posts:
- son solo titulo con: imágenes / videos /historietas / links a podcast/ tests
- son un titulo con un link que redirecciona al verdadero ensayo

### 2. Duplicados y ultimas versiones



In [3]:
# Duplicados exactos de document_text
dupes = df[df["document_text"].duplicated(keep=False)].sort_values("post_date")
print(f"Filas con document_text duplicado: {len(dupes)}")
display(dupes[["ID", "post_date", "title_text", "word_count", "source_url"]])

Filas con document_text duplicado: 2


,ID,post_date,title_text,word_count,source_url
707,40375,2016-03-14 10:34:40,"Escrituras del yo, de lo personal a lo colectivo",661,https://www.revistaanfibia.com/escrituras-del-yo-de-lo-personal-a-lo-colectivo/
862,46921,2016-07-22 11:00:37,"Escrituras del yo, de lo personal a lo colectivo",661,https://www.revistaanfibia.com/escrituras-del-yo-de-lo-personal-a-lo-colectivo-2/


Se nota en este caso y en otros vistos que cuando se republica o se lanza otra version se el agrega un -N/ al url. Por lo que vamos a ver si encontramos archivos redundantes

In [4]:
#display de los post que en la antepenultima parte del url tienen un -numero/
versiones= df[df["source_url"].str.contains(r"-\d+/")].sort_values("post_date")
print(f"Filas con source_url que contienen '-<número>/': {len(versiones)}")
display(versiones[["ID", "post_date", "title_text", "word_count", "source_url"]]) 

Filas con source_url que contienen '-<número>/': 90


,ID,post_date,title_text,word_count,source_url
73,1882,2013-12-02 17:12:24,Mis días con Giselle Rímolo en la cárcel,3765,https://www.revistaanfibia.com/mis-dias-con-giselle-rimolo-en-la-carcel-2/
82,1704,2014-03-19 16:19:00,El videojuego de Dios,5412,https://www.revistaanfibia.com/el-videojuego-de-dios-2/
174,2300,2014-07-07 18:08:42,Una mujer puerca y un Dios personal,490,https://www.revistaanfibia.com/una-mujer-puerca-y-un-dios-personal-2/
197,2981,2014-08-04 11:28:45,"No fui yo, fue mamá",6185,https://www.revistaanfibia.com/no-fui-yo-fue-mama-2/
296,15180,2014-12-02 12:51:45,Roller Derby: Sentirte poderosa,3246,https://www.revistaanfibia.com/roller-derby-sentirte-poderosa-2/
...,...,...,...,...,...
3001,190528,2023-10-04 12:02:15,"Enemigo de Allende, opositor a Pinochet, frenteamplista y peronista",3540,https://www.revistaanfibia.com/thieme-enemigo-de-allende-opositor-a-pinochet-frenteamplista-y-peronista-4/
3424,205296,2025-07-25 09:05:34,Nos persiguen los 90,2427,https://www.revistaanfibia.com/debilidad-humana-lila-siegrist-nos-persiguen-los-90/
3437,205579,2025-08-12 13:01:43,Un oasis entre el cemento y los agroquímicos,4564,https://www.revistaanfibia.com/un-oasis-entre-el-cemento-y-los-agroquimicos-2/
3618,220810,2026-06-10 08:46:07,El bien y el mal definen por penal,3002,https://www.revistaanfibia.com/el-bien-y-el-mal-definen-por-penal-mundial-2026/


In [5]:
# de los que se encontraron vemos si hay otro link de una version anterior
import re

# Para cada URL con -numero/ extraer la parte anterior al -numero/
def slug_sin_sufijo(url):
    match = re.search(r"(https://www\.revistaanfibia\.com/.+)-\d+/$", url)
    if match:
        return match.group(1) + "/"
    return None

versiones["url_sin_sufijo"] = versiones["source_url"].map(slug_sin_sufijo)

# Ver si esa URL existe en el dataset completo
urls_existentes = set(df["source_url"])
versiones["original_existe"] = versiones["url_sin_sufijo"].isin(urls_existentes)
print(f"\nCon original en el dataset: {versiones['original_existe'].sum()}")
print(f"Sin original: {(~versiones['original_existe']).sum()}")




Con original en el dataset: 9
Sin original: 81


In [6]:
# Imprimo los que tienen el original en el dataset y que tienen el mismo title_text
versiones_con_original = versiones[versiones["original_existe"]]
display(versiones_con_original[["ID", "post_date", "title_text", "source_url", "url_sin_sufijo"]])


,ID,post_date,title_text,source_url,url_sin_sufijo
417,23476,2015-05-15 11:42:57,Seminario de Periodismo Político,https://www.revistaanfibia.com/seminario-de-periodismo-politico-2/,https://www.revistaanfibia.com/seminario-de-periodismo-politico/
584,32155,2015-10-27 12:33:20,Jornada: Argentina después de las elecciones,https://www.revistaanfibia.com/argentina-despues-de-las-elecciones-2/,https://www.revistaanfibia.com/argentina-despues-de-las-elecciones/
862,46921,2016-07-22 11:00:37,"Escrituras del yo, de lo personal a lo colectivo",https://www.revistaanfibia.com/escrituras-del-yo-de-lo-personal-a-lo-colectivo-2/,https://www.revistaanfibia.com/escrituras-del-yo-de-lo-personal-a-lo-colectivo/
915,49098,2016-09-12 14:41:06,Caja de Herramientas para Nuevos Periodistas,https://www.revistaanfibia.com/caja-de-herramientas-para-nuevos-periodistas-2/,https://www.revistaanfibia.com/caja-de-herramientas-para-nuevos-periodistas/
1123,65048,2017-04-21 11:50:40,Periodismo y Cultura Gastronómica,https://www.revistaanfibia.com/periodismo-y-cultura-gastronomica-2/,https://www.revistaanfibia.com/periodismo-y-cultura-gastronomica/
1311,84267,2017-10-24 09:20:17,Crónicas en el Viejo Hotel Ostende,https://www.revistaanfibia.com/cronicas-viejo-hotel-ostende-2/,https://www.revistaanfibia.com/cronicas-viejo-hotel-ostende/
1410,95047,2018-02-02 15:03:32,Talleres y Seminarios Anfibia 2018,https://www.revistaanfibia.com/talleres-y-seminarios-anfibia-2018/,https://www.revistaanfibia.com/talleres-y-seminarios-anfibia/
1615,106655,2018-07-27 20:50:46,Wonder gorda,https://www.revistaanfibia.com/wonder-gorda-2/,https://www.revistaanfibia.com/wonder-gorda/
2961,188909,2023-08-25 10:36:09,Hacia una valoración de la diversidad,https://www.revistaanfibia.com/hacia-una-valoracion-de-la-diversidad-2/,https://www.revistaanfibia.com/hacia-una-valoracion-de-la-diversidad/


Salvo el url de los talleres que tiene sentido que sea versionado, los demas son versiones viejas de un articulo que se volvió a publicar. Queda como decision ver si eliminarlos o dejarlos. Lo mas consistente seria no utilizar las versiones viejas

### 3. Detección de idioma

Se usa `langdetect` sobre los primeros 500 caracteres de `body_text`. 

In [7]:
from langdetect import detect, LangDetectException

def detect_lang(text):
    if not text or len(str(text).strip()) < 50:
        return None
    try:
        return detect(str(text)[:500])
    except LangDetectException:
        return None

df["lang"] = df["body_text"].map(detect_lang)
print(df["lang"].value_counts().to_string())

lang
es    3562
fr      39
en       4
pt       2


In [8]:
non_spanish = df[df["lang"].notna() & df["lang"].ne("es")]
print(f"Posts no detectados como español: {len(non_spanish)}")
display(non_spanish[["ID", "post_date", "title_text", "lang", "source_url"]].head(20))

Posts no detectados como español: 45


,ID,post_date,title_text,lang,source_url
336,15877,2015-01-27 15:43:56,Santuario Lemebel,pt,https://www.revistaanfibia.com/homenaje-santuario-lemebel/
348,16153,2015-02-24 13:49:57,Nisman’s puzzle,en,https://www.revistaanfibia.com/nismans-puzzle/
407,22978,2015-05-05 13:04:22,Le puzzle Nisman,fr,https://www.revistaanfibia.com/le-puzzle-nisman/
408,22987,2015-05-05 13:33:51,Le voisin musulman,fr,https://www.revistaanfibia.com/le-voisin-musulman/
410,23054,2015-05-07 09:02:41,Qu’a fait Nisman dans l’affaire AMIA?,fr,https://www.revistaanfibia.com/quafait-nisman-dans-laffaire-amia/
420,23750,2015-05-21 12:15:38,"Laclau, le provocateur",fr,https://www.revistaanfibia.com/laclau-le-provocateur/
423,23939,2015-05-28 08:15:16,Quel serait l’héritage de Laclau en Argentine?,fr,https://www.revistaanfibia.com/quel-serait-lheritage-de-laclau-en-argentine/
467,25708,2015-06-26 12:09:34,LES MACHOS M’APPELLENT «FEMINAZI»,fr,https://www.revistaanfibia.com/les-machos-mappellent-feminazi/
469,25860,2015-07-01 15:55:17,Le genre d’une décennie,fr,https://www.revistaanfibia.com/le-genre-dune-decennie/
482,26782,2015-07-10 16:07:28,Plus rien ne sera pareil,fr,https://www.revistaanfibia.com/plus-rien-ne-sera-pareil/


In [9]:
print("Posts con body en portugués:", len(df[df["lang"] == "pt"]))
display(df[df["lang"] == "pt"][["ID", "post_date", "title_text", "word_count", "source_url"]].sort_values("post_date"))

Posts con body en portugués: 2


,ID,post_date,title_text,word_count,source_url
336,15877,2015-01-27 15:43:56,Santuario Lemebel,92,https://www.revistaanfibia.com/homenaje-santuario-lemebel/
2085,145747,2020-04-13 11:11:43,Muy en una: Edición Corona Virus,145,https://www.revistaanfibia.com/muy-en-una-edicion-corona-virus/


Viendolo a mano se ve que, a diferencia de los demás idiomas, los que langdetect identifica como portugues estan en español. Se trata de un error, sin embargo si detectó bien los post en francés y inglés.

### 4. Decisión sobre `document_text`

Actualmente `document_text = title_text + ". " + body_text`.

- El copete es un resumen corto pero que tiene información relevante,cuando existe. Seria útil incorporarlo,  principalmente para tener texto en los post que solo tienen imagenes. Lo mismo sucede con 'tagline' .
Se genera una versión extendida de document_text para agregar esta información.

In [10]:
copete_present = df["copete"].fillna("").str.strip().ne("")
print(f"Posts con copete: {copete_present.sum()} de {len(df)} ({copete_present.mean():.1%})")

Posts con copete: 3501 de 3622 (96.7%)


In [11]:
def build_document_text(row):
    parts = [str(row["title_text"]).strip()]
    copete = str(row.get("copete", "") or "").strip()
    if copete:
        parts.append(copete)
    tagline = str(row.get("tagline", "") or "").strip()
    if tagline:
        parts.append(tagline)
    body = str(row["body_text"]).strip()
    if body:
        parts.append(body)
    return ". ".join(p.rstrip(".") for p in parts if p)

df["document_text_extended"] = df.apply(build_document_text, axis=1)
df["word_count_extended"] = df["document_text_extended"].str.findall(r"(?u)\b\w+\b").str.len()


# Comparacion de longitud
diff = df["word_count_extended"] - df["word_count"]
print(f"Palabras adicionales por incluir copete y tagline: {diff.sum()}")

Palabras adicionales por incluir copete y tagline: 308000


### 5. Corpus final

Aplicar todos los filtros y exportar. Ajustar las variables al inicio según lo que se vio arriba.

In [12]:
import re

# ── Decisiones del analisis ──────────────────────────────────────────────────
INCLUDE_COPETE   = True  # agregar copete y tagline a document_text
FILTER_LANGUAGE  = True  # excluir ingles y frances (pt queda: langdetect confunde con es)
EXCLUDE_VERSIONS = True  # excluir versiones viejas (conserva la mas actual)

# ── Columnas de texto segun decision de copete ───────────────────────────────
text_col = "document_text_extended" if INCLUDE_COPETE else "document_text"
wc_col   = "word_count_extended"    if INCLUDE_COPETE else "word_count"

# ── IDs a excluir: originales que tienen una version mas nueva en el dataset ─
def slug_sin_sufijo(url):
    match = re.search(r"(https://www\.revistaanfibia\.com/.+)-\d+/$", url)
    return match.group(1) + "/" if match else None

versiones_mask = df["source_url"].str.contains(r"-\d+/", regex=True, na=False)
versiones_temp = df[versiones_mask].copy()
versiones_temp["url_original"] = versiones_temp["source_url"].map(slug_sin_sufijo)

# URLs originales que tienen al menos una version mas nueva
urls_con_version_nueva = set(versiones_temp["url_original"].dropna())

# Excluir los posts cuya URL es el original reemplazado por una version mas nueva
ids_version_vieja = set(
    df.loc[df["source_url"].isin(urls_con_version_nueva), "ID"]
)
print(f"IDs excluidos por tener version mas nueva: {len(ids_version_vieja)}")

# ── Mascara principal ────────────────────────────────────────────────────────
mask = (
    df["body_text"].fillna("").str.strip().ne("")     # sin body vacio
    & ~df["document_text"].duplicated(keep="last")  # sin duplicados exactos (conserva el mas nuevo)
    & df["post_date"].notna()                          # descarta fechas corruptas o ausentes
)
if FILTER_LANGUAGE:
    mask &= df["lang"].isin(["es", "pt"])
if EXCLUDE_VERSIONS:
    mask &= ~df["ID"].isin(ids_version_vieja)

# ── Corpus final ─────────────────────────────────────────────────────────────
corpus = df[mask].copy()
corpus["document_text_final"] = corpus[text_col]

print(f"Corpus final: {len(corpus):,} documentos")
print(f"Rango: {corpus.post_date.min():%Y-%m-%d} a {corpus.post_date.max():%Y-%m-%d}")
display(corpus[["ID", "post_date", "title_text", wc_col]].head(10))

CLEAN_CSV = PROCESSED_PATH / "corpus_anfibia_limpio.csv"
corpus.to_csv(CLEAN_CSV, index=False, encoding="utf-8-sig")

print(f"Exportado: {len(corpus):,} documentos")
print(f"  {CLEAN_CSV}")

IDs excluidos por tener version mas nueva: 9
Corpus final: 3,556 documentos
Rango: 2012-05-14 a 2026-06-16


,ID,post_date,title_text,word_count_extended
0,15668,2012-05-14 13:43:22,La China invisible,4898
1,15706,2012-05-24 13:36:12,El rico que se cree Steve Jobs,3234
2,15743,2012-06-05 14:52:58,La vaca sagrada,4973
3,15695,2012-06-14 10:16:11,Los arruinados del Paraná,4123
4,15687,2012-06-25 14:52:05,De Chile a Tacna: una historia de carniceros y cirujanos,2040
5,14466,2012-06-28 14:19:51,Bolivia en llamas,3240
6,14454,2012-07-06 13:27:04,Los líderes de la primavera mexicana (I),5145
7,3650,2012-07-16 17:08:34,Nunca te metas con las mariscadoras de Vigo,2545
8,3669,2012-07-23 17:32:48,"Lucía, la remadora olímpica de las manos imposibles",1417
9,3639,2012-07-30 15:38:38,El fisicoculturista que no puede mirarse en el espejo,3526


Exportado: 3,556 documentos
  C:\Trabajos\anfibia-unsam\data\processed\corpus_anfibia_limpio.csv


## 6. Lematización

Este proceso toma el corpus limpio recién filtrado y aplica lematización con spaCy (`es_core_news_md`).

El objetivo es reducir las palabras a su forma base (por ejemplo: "elecciones" -> "elección", "fueron" -> "ser") para optimizar los modelos de tópicos posteriores (Bag of Words y BERTopic).

Salida: `data/processed/corpus_anfibia_lematizado.csv`.

In [14]:
import spacy
import warnings

warnings.filterwarnings("ignore", category=UserWarning)

try:
    nlp = spacy.load("es_core_news_md", disable=["parser", "ner"])
    SPACY_AVAILABLE = True
except OSError:
    SPACY_AVAILABLE = False
    print("Falta el modelo de spaCy. Ejecutá en terminal: python -m spacy download es_core_news_md")

print(f"spaCy disponible: {SPACY_AVAILABLE}")

spaCy disponible: True


### 6.1 Preparación de columnas y doc_id

In [15]:
TEXT_COLUMN = "document_text_final"
ATTRIBUTE_COLUMNS = ["tagline", "autor_names", "post_date", "post_year", "category", "post_tag", "text_type"]
ID_COLUMNS = ["ID", "source_url", "title_text"]
USE_COLUMNS = [c for c in ID_COLUMNS + ATTRIBUTE_COLUMNS + [TEXT_COLUMN] if c in corpus.columns]

df_lem = corpus[USE_COLUMNS].copy()

df_lem[TEXT_COLUMN] = df_lem[TEXT_COLUMN].fillna("").astype(str)
for col in ATTRIBUTE_COLUMNS:
    if col in df_lem.columns:
        df_lem[col] = df_lem[col].fillna("").astype(str).str.strip()

df_lem = df_lem[df_lem[TEXT_COLUMN].str.strip().ne("")].copy()
df_lem = df_lem.reset_index(drop=True)
df_lem.insert(0, "doc_id", df_lem.index)

print(f"Documentos a procesar: {df_lem.shape[0]}")
display(df_lem.head(3))

Documentos a procesar: 3556


,doc_id,ID,source_url,title_text,tagline,autor_names,post_date,post_year,category,post_tag,text_type,document_text_final
0,0,15668,https://www.revistaanfibia.com/la-china-invisible/,La China invisible,Feria La Salada,Sebastián Hacher; Ariel Wilkis,2012-05-14 13:43:22,2012,Política y Economía,,Crónica,"La China invisible. En La Ribera, la última de las cuatro ferias de La Salada, un orden invisible permite que una ci..."
1,1,15706,https://www.revistaanfibia.com/grobocopatel-el-rico-que-se-cree-steve-jobs/,El rico que se cree Steve Jobs,Gustavo Grobocopatel,Graciela Mochkofsky; Alexandre Roig,2012-05-24 13:36:12,2012,Política y Economía,Perfil político,Crónica,"El rico que se cree Steve Jobs. Gustavo Grobocopatel se presenta como el gran modernizador del campo argentino, un s..."
2,2,15743,https://www.revistaanfibia.com/la-vaca-sagrada/,La vaca sagrada,,Josefina Licitra,2012-06-05 14:52:58,2012,Tecnologías,,,"La vaca sagrada. A 400 kilómetros de Buenos Aires, protegida por guardias, alarmas y sensores láser vive Rosita, una..."


### 6.2 Procesamiento y Lematización en lotes

Lematizamos usando `nlp.pipe` por lotes para optimizar velocidad y consumo de memoria. Incluye la corrección ad-hoc del término "país/países".

In [16]:
lemma_corrections = {
    "paises": "país",
    "países": "país"
}

def lematizar_textos(textos, batch_size=100):
    textos_lematizados = []
    for doc in nlp.pipe(textos, batch_size=batch_size):
        lemas = []
        for token in doc:
            if not token.is_punct and not token.is_space:
                lemma = token.lemma_.lower()
                lemma = lemma_corrections.get(lemma, lemma)
                lemas.append(lemma)
        textos_lematizados.append(" ".join(lemas))
    return textos_lematizados

df_lem["document_text_lemmatized"] = lematizar_textos(df_lem[TEXT_COLUMN].tolist())

display(df_lem[[TEXT_COLUMN, "document_text_lemmatized"]].head(5))

,document_text_final,document_text_lemmatized
0,"La China invisible. En La Ribera, la última de las cuatro ferias de La Salada, un orden invisible permite que una ci...",el china invisible en el ribera el último de el cuatro feria de el salada uno orden invisible permitir que uno ciuda...
1,"El rico que se cree Steve Jobs. Gustavo Grobocopatel se presenta como el gran modernizador del campo argentino, un s...",el rico que él creer steve jobs gustavo grobocopatel él presentar como el gran modernizador del campo argentino uno ...
2,"La vaca sagrada. A 400 kilómetros de Buenos Aires, protegida por guardias, alarmas y sensores láser vive Rosita, una...",el vaca sagrado a 400 kilómetro de buenos aires protegido por guardias alarma y sensor láser vivir rosita uno vaca ú...
3,"Los arruinados del Paraná. Cuando el “casino más lujoso de Sudamérica” llegó a Victoria, una pequeña ciudad de Entre...",el arruinado del paraná cuando el casino más lujoso de sudamérica llegar a victoria uno pequeño ciudad de entre ríos...
4,De Chile a Tacna: una historia de carniceros y cirujanos. Miles de mujeres chilenas cruzan la frontera para ir en bu...,de chile a tacna uno historia de carnicero y cirujano mil de mujer chileno cruzar el frontera para ir en busca de un...


### 6.3 Guardado del corpus lematizado

In [17]:
# Guardamos el original y actualizamos la columna de texto principal
df_lem["document_text_original"] = df_lem[TEXT_COLUMN]
df_lem[TEXT_COLUMN] = df_lem["document_text_lemmatized"]
df_lem = df_lem.drop(columns=["document_text_lemmatized"])

LEMMATIZED_CSV = PROCESSED_PATH / "corpus_anfibia_lematizado.csv"
df_lem.to_csv(LEMMATIZED_CSV, index=False, encoding="utf-8-sig")

print(f"Corpus lematizado guardado exitosamente en:")
print(f"  {LEMMATIZED_CSV}")

Corpus lematizado guardado exitosamente en:
  ..\data\processed\corpus_anfibia_lematizado.csv
